## Carregando Dados e Bibliotecas necessárias.

In [1]:
import pandas as pd
import numpy as np

data_layer_filepath = '../../data_layer/'

df = pd.read_csv(data_layer_filepath + 'raw/airbnb-dataset.csv', low_memory=False)
print("Dataset carregado com sucesso!")
df.head()

Dataset carregado com sucesso!


,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,country,...,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,house_rules,license
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,$193,10.0,9.0,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,NaN
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,$28,30.0,45.0,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,NaN
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,NaN,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,...,$124,3.0,0.0,NaN,NaN,5.0,1.0,352.0,"I encourage you to use my kitchen, cooking and...",NaN
3,1002755,NaN,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,...,$74,30.0,270.0,7/5/2019,4.64,4.0,1.0,322.0,NaN,NaN
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,...,$41,10.0,9.0,11/19/2018,0.10,3.0,1.0,289.0,"Please no smoking in the house, porch or on th...",NaN


# Tratamento dos Dados.

## Padronização dos Nomes das Colunas.

In [2]:
df.rename(
    columns={col: col.lower().replace(' ', '_') for col in df.columns},
    inplace=True
)
print(df.columns)

Index(['id', 'name', 'host_id', 'host_identity_verified', 'host_name',
       'neighbourhood_group', 'neighbourhood', 'lat', 'long', 'country',
       'country_code', 'instant_bookable', 'cancellation_policy', 'room_type',
       'construction_year', 'price', 'service_fee', 'minimum_nights',
       'number_of_reviews', 'last_review', 'reviews_per_month',
       'review_rate_number', 'calculated_host_listings_count',
       'availability_365', 'house_rules', 'license'],
      dtype='object')


## Remoção de Colunas Desnecessárias

Como quase todas as tuplas de **license** estavam como nulas, optamos por não trabalhar com essa coluna. Além disso, optamos por remover as colunas de **Country** e **Country_code** visto que sabemos que todas se enquadram no Estados Unidos e possuem o códido do país como "US".

In [3]:
cols_to_drop = ['country', 'country_code', 'license']
df.drop(columns=cols_to_drop, inplace=True)

for col in cols_to_drop:
    if col not in df.columns:
        print(f"Coluna {col} deletada!")

print("Colunas: ")
print(df.columns)

Coluna country deletada!
Coluna country_code deletada!
Coluna license deletada!
Colunas: 
Index(['id', 'name', 'host_id', 'host_identity_verified', 'host_name',
       'neighbourhood_group', 'neighbourhood', 'lat', 'long',
       'instant_bookable', 'cancellation_policy', 'room_type',
       'construction_year', 'price', 'service_fee', 'minimum_nights',
       'number_of_reviews', 'last_review', 'reviews_per_month',
       'review_rate_number', 'calculated_host_listings_count',
       'availability_365', 'house_rules'],
      dtype='object')


## Correções dos Tipos de Dados.

1) **price** e **service_fee** possuem o caracter especial "$" e estão como object. Com isso, iremos altera-las para o tipo númerico (Float)

In [4]:
money_columns = ['price', 'service_fee']

print("Tipos de dados antes da correção:")
print(df[money_columns].dtypes)

for col in money_columns:
    df[col] = df[col].str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip().astype(float)

print("Tipos de dados corrigidos:")
print(df[money_columns].dtypes)

Tipos de dados antes da correção:
price          object
service_fee    object
dtype: object
Tipos de dados corrigidos:
price          float64
service_fee    float64
dtype: object


2) **construction_year**, **number_of_reviews**, **availability_365** e **calculated_host_listings_count**: Não faria sentido estar sendo guardado em Float já que era o ano de construção. Ou seja, nunca viria um número decimal. O mesmo vale para availability_365, number_of_reviews e para calculated_host_listings_count.

In [5]:
floats_to_ints = ['availability_365', 'construction_year','calculated_host_listings_count', 'number_of_reviews']

for col in floats_to_ints:
    print(f"Tipo de dado da chave {col} antes da correção: {df[col].dtype}")

    df.dropna(subset=[col], inplace=True)
    df[col] = df[col].astype('int64')

    print(f"Tipo de dado da chave {col} após a correção: {df[col].dtype}")

Tipo de dado da chave availability_365 antes da correção: float64
Tipo de dado da chave availability_365 após a correção: int64
Tipo de dado da chave construction_year antes da correção: float64
Tipo de dado da chave construction_year após a correção: int64
Tipo de dado da chave calculated_host_listings_count antes da correção: float64
Tipo de dado da chave calculated_host_listings_count após a correção: int64
Tipo de dado da chave number_of_reviews antes da correção: float64
Tipo de dado da chave number_of_reviews após a correção: int64


3) **host_identity_verified**: A coluna host_identity_verified pode ser transformada em booleano (True/False), o que é mais eficiente e semanticamente correto.

In [6]:
df['host_identity_verified'] = df['host_identity_verified'].astype(str).str.lower().str.strip()

to_replace = {
    'verified': True,
    'unconfirmed': False
}

df['host_identity_verified'] = df['host_identity_verified'].replace(to_replace).astype(bool)


print("Tipo de dado da coluna APÓS o tratamento:")
print(df['host_identity_verified'].dtype)
print("\nValores únicos na coluna APÓS o tratamento:")
print(df['host_identity_verified'].unique())
print("\nContagem de valores na coluna APÓS o tratamento:")
print(df['host_identity_verified'].value_counts(dropna=False))

Tipo de dado da coluna APÓS o tratamento:
bool

Valores únicos na coluna APÓS o tratamento:
[False  True]

Contagem de valores na coluna APÓS o tratamento:
host_identity_verified
True     50848
False    50664
Name: count, dtype: int64


4. **Colunas object**: algumas colunas tem o tipo misto `object`. Na análise realizada na camada bronze, concluímos que essas colunas devem ter o tipo `string`. Além disso, a coluna `instant_bookable` deveria ter o tipo `bool`. Por fim, a coluna `last_review` deveria ter um tipo de dados que melhor representa uma data.

In [7]:
string_cols = [
    'name',
    'host_name',
    'neighbourhood_group',
    'neighbourhood',
    'cancellation_policy',
    'room_type',
    'house_rules',
]

for col in string_cols:
    df[col] = df[col].astype('string')

df['instant_bookable'] = df['instant_bookable'].astype(bool)

df['last_review'] = pd.to_datetime(df['last_review'])

5. **minimum_nights**: esta coluna é do tipo `float64`, mas deveria ter um tipo inteiro, visto que trata da quantidade de noites mínimas que um interessado deve passar num lugar anunciado.

In [8]:
df.dropna(subset=['minimum_nights'], inplace=True)
df['minimum_nights'] = df['minimum_nights'].astype('int64')

## Correção de Inconsistência nos Dados.

### Correção nos erros de digitação no nome dos bairros que apresentavam "brookln" e "manhatan"

In [9]:

df['neighbourhood_group'] = df['neighbourhood_group'].replace({
    'brookln': 'Brooklyn',
    'manhatan': 'Manhattan'
})


print(df['neighbourhood_group'].unique())

<StringArray>
['Brooklyn', 'Manhattan', <NA>, 'Queens', 'Staten Island', 'Bronx']
Length: 6, dtype: string


### Tratamento de Valores Ausentes.


1) Remoção de Anúncios sem preço.

In [10]:

df.dropna(subset=['price', 'service_fee'], inplace=True)

print(f"Valores nulos em 'price' após remoção: {df['price'].isnull().sum()}")

Valores nulos em 'price' após remoção: 0


2) Criação de Coluna booleana para house_rules: Como metade dos valores é nulo, iremos criar uma nova coluna para indicar se essa "casa" possui ou não regras definidas.

In [11]:

df['has_house_rules'] = df['house_rules'].notna()

# Podemos agora remover a coluna original se o conteúdo de texto não for usado
# df_silver.drop(columns=['house_rules'], inplace=True)


print(df['has_house_rules'].value_counts())

has_house_rules
False    51174
True     49484
Name: count, dtype: int64


3) Preenchimento dos poucos anúncios sem nome (ou sem nome de host) com "Sem nome informado"

In [12]:
for col in ['name', 'host_name']:
    df[col] = df[col].fillna('Sem nome informado')

4) Remoção dos demais **nans**

In [13]:
nans_to_drop = [
    'neighbourhood', 
    'neighbourhood_group', 
    'lat', 
    'long',
    'host_identity_verified',
    'minimum_nights',
    'review_rate_number',
]

df.dropna(subset=nans_to_drop, inplace=True)

## Tratamentos de Valores invalidados.

Foram identificado alguns valores negativos na coluna **availability_365**. No entanto, não faz sentido que essa coluna tenha valores negativos, visto que ela indica uma quantidade de dias futuros. Ou seja: o valor mínimo deveria ser 0.

In [14]:
df = df[df['availability_365'] >= 0]
print(df['availability_365'].min())

0


# Tratamentos de Dados Duplicados


In [15]:
print("--- DIAGNÓSTICO E REMOÇÃO DE DUPLICADOS ---")
print(f"Número de linhas TOTAIS no DataFrame inicial: {len(df)}")
print("-" * 45)


duplicated_lines = df[df.duplicated(subset=['id'], keep=False)]

if not duplicated_lines.empty:
    num_of_duplicated_ids = duplicated_lines['id'].nunique()
    print(f"Encontrados {num_of_duplicated_ids} IDs que se repetem.")
    print(f"Esses IDs correspondem a um total de {len(duplicated_lines)} linhas no DataFrame.")
else:
    print("Não existem IDs duplicados no dataset.")

df_without_duplicates = df.drop_duplicates(subset=['id'], keep='first')

print(f"\nNúmero de linhas APÓS a remoção: {len(df_without_duplicates)}")
print(f"Cálculo da operação: {len(df)} (linhas iniciais) - {df.duplicated(subset=['id']).sum()} (ocorrências extras) = {len(df_without_duplicates)}")
print("-" * 45)

number_of_duplicates_after_cleanup = df_without_duplicates.duplicated(subset=['id']).sum()

print("VERIFICAÇÃO FINAL:")
print(f"Número de IDs duplicados no DataFrame final: {number_of_duplicates_after_cleanup}")

df = df_without_duplicates

if df.duplicated(subset=['id']).sum() == 0:
    print("\n✅ SUCESSO! A remoção de duplicados foi confirmada com sucesso.")
else:
    print("\n⚠️ ATENÇÃO! Ainda existem duplicados no DataFrame.")


--- DIAGNÓSTICO E REMOÇÃO DE DUPLICADOS ---
Número de linhas TOTAIS no DataFrame inicial: 99986
---------------------------------------------
Encontrados 524 IDs que se repetem.
Esses IDs correspondem a um total de 1048 linhas no DataFrame.

Número de linhas APÓS a remoção: 99462
Cálculo da operação: 99986 (linhas iniciais) - 524 (ocorrências extras) = 99462
---------------------------------------------
VERIFICAÇÃO FINAL:
Número de IDs duplicados no DataFrame final: 0

✅ SUCESSO! A remoção de duplicados foi confirmada com sucesso.


## Dicionario - Silver

In [16]:
print("--- Amostra Aleatória de 20 Linhas do DataFrame 'df_silver' ---")

# O .sample(20) pega 20 linhas aleatórias do DataFrame.
# O .reset_index(drop=True) é para a visualização ficar mais limpa, sem o índice antigo.
display(df.sample(20).reset_index(drop=True))


print("\n\n--- Resumo das Informações (Info) ---")
df.info()

--- Amostra Aleatória de 20 Linhas do DataFrame 'df_silver' ---


,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,instant_bookable,...,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules,has_house_rules
0,29186862,New York Best Choice New Large Apartment bySu...,30944116706,False,Uri,Brooklyn,Midwood,40.62447,-73.96140,True,...,206.0,2,67,2022-02-27,4.93,4.0,3,271,<NA>,False
1,33732291,Charming Apt in the Heart of SoHo,9276555527,True,Hugo,Manhattan,SoHo,40.72384,-74.00272,True,...,96.0,2,1,2022-01-02,0.48,2.0,1,219,<NA>,False
2,3431456,Perfect NYC Flat! Modern!,94100844133,True,AFI Apartments,Manhattan,Upper East Side,40.77233,-73.95720,False,...,196.0,30,1,2017-02-04,0.03,3.0,29,98,"no smoking allowed, leave it how you found it.",True
3,50764671,Nice and spacious 3BR home in Queens,42476196189,True,David,Queens,East Elmhurst,40.76170,-73.89134,True,...,234.0,2,32,2019-01-01,1.03,3.0,1,55,<NA>,False
4,21698226,Affordable Room near JFK,87493560699,True,Portia,Queens,Jamaica,40.67385,-73.77874,True,...,149.0,1,1,2019-06-23,1.00,4.0,4,253,We just ask that you are clean. If you are her...,True
5,20751584,Boerum Hill 2 BR Suite in Classic NYC Brownstone,14172926149,False,Allen,Brooklyn,Boerum Hill,40.68528,-73.98634,True,...,120.0,1,9,2019-06-09,0.89,3.0,3,25,<NA>,False
6,36439113,Lovely Apartment 1bd Clean & Quiet. You'll lov...,73549381977,False,Ivan,Brooklyn,Bay Ridge,40.63132,-74.02797,False,...,145.0,30,1,2021-08-31,0.14,3.0,1,0,<NA>,False
7,30405236,Cozy Quiet Room in the Big Apple on Broadway!!!,59765116345,False,Mark & Will,Manhattan,Harlem,40.82953,-73.94923,False,...,20.0,7,203,2022-02-18,3.14,2.0,2,155,<NA>,False
8,1616046,Gorgeous Summer Duplex/Yard sublet,9729014487,False,Verena And Dylan,Brooklyn,Bedford-Stuyvesant,40.68229,-73.94287,True,...,191.0,14,3,2013-09-01,0.04,4.0,1,338,<NA>,False
9,44800383,Cozy Studio on the Upper East Side!,73651693636,True,Kara,Manhattan,Upper East Side,40.76138,-73.95980,True,...,115.0,30,1,2017-09-30,0.05,1.0,121,336,Check-in time is 1PM. Read the manual after yo...,True




--- Resumo das Informações (Info) ---
<class 'pandas.core.frame.DataFrame'>
Index: 99462 entries, 0 to 102044
Data columns (total 24 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   id                              99462 non-null  int64         
 1   name                            99462 non-null  string        
 2   host_id                         99462 non-null  int64         
 3   host_identity_verified          99462 non-null  bool          
 4   host_name                       99462 non-null  string        
 5   neighbourhood_group             99462 non-null  string        
 6   neighbourhood                   99462 non-null  string        
 7   lat                             99462 non-null  float64       
 8   long                            99462 non-null  float64       
 9   instant_bookable                99462 non-null  bool          
 10  cancellation_policy             99

## Salvando Dataset

In [17]:

df.to_csv(data_layer_filepath + 'silver/airbnb-dataset-silver.csv', index=False)

print("Dataset da camada Silver salvo com sucesso!")

Dataset da camada Silver salvo com sucesso!


## Carregando os dados para na base de dados

In [18]:
import pandas as pd
from psycopg import connect, sql

print("--- Iniciando processo de carga e verificação no PostgreSQL ---")

DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5433"
DB_NAME = "airbnb"
DB_SCHEMA = "silver"
TABLE_NAME = "listings"
TABLE_FULL_NAME = f"{DB_SCHEMA}.{TABLE_NAME}"

DDL = f"""
CREATE SCHEMA IF NOT EXISTS {DB_SCHEMA};
DROP TABLE IF EXISTS {TABLE_FULL_NAME};
CREATE TABLE {TABLE_FULL_NAME} (
    id BIGINT PRIMARY KEY, name TEXT, host_id BIGINT, host_identity_verified BOOLEAN,
    host_name VARCHAR(255), neighbourhood_group VARCHAR(255), neighbourhood VARCHAR(255),
    lat NUMERIC(10, 7), long NUMERIC(10, 7), instant_bookable BOOLEAN,
    cancellation_policy VARCHAR(100), room_type VARCHAR(100), construction_year INTEGER,
    price NUMERIC(10, 2), service_fee NUMERIC(10, 2), minimum_nights INTEGER,
    number_of_reviews INTEGER, last_review DATE, reviews_per_month NUMERIC(5, 2),
    review_rate_number NUMERIC(10, 2), calculated_host_listings_count INTEGER,
    availability_365 INTEGER, house_rules TEXT, has_house_rules BOOLEAN
);
"""

print(f"Total de linhas a serem carregadas: {len(df)}")

cols = list(df.columns)

insert_query = sql.SQL("INSERT INTO {} ({}) VALUES ({})").format(
    sql.Identifier(DB_SCHEMA, TABLE_NAME),                      
    sql.SQL(", ").join(map(sql.Identifier, cols)),               
    sql.SQL(", ").join(sql.Placeholder() * len(cols))            
)

with connect(f"host={DB_HOST} dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} port={DB_PORT}") as conn:
    print("\nConexão com o PostgreSQL estabelecida.")
    with conn.cursor() as cur:
        cur.execute(DDL)
        print("Estrutura do banco de dados criada com sucesso.")
        conn.commit()

        print("Iniciando carga de dados...")

        for _, row in df.iterrows():
            values = [None if pd.isna(v) else v for v in row]
            cur.execute(insert_query, values)

        conn.commit()
        print("Carga concluída com sucesso!")


--- Iniciando processo de carga e verificação no PostgreSQL ---
Total de linhas a serem carregadas: 99462

Conexão com o PostgreSQL estabelecida.
Estrutura do banco de dados criada com sucesso.
Iniciando carga de dados...
Carga concluída com sucesso!
